In [1]:
import sys
from pathlib import Path

root_dir = Path.cwd().parent 
if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))

import pandas as pd
import io
from utils.connections import get_azure_connection, get_postgres_connection

In [2]:
with get_azure_connection() as conn:
    with conn.cursor() as cursor:
        cursor.execute("""
            SELECT 
                a.*,
                'dbo.nowrthwind' as origem
            FROM orders a;
            """)
        rows= cursor.fetchall()
        columns = [column[0] for column in cursor.description]
        df = pd.DataFrame(rows, columns = columns)

In [4]:
with get_postgres_connection() as conn:
    with conn.cursor() as cursor:
        # Limpa a tabela antes da nova carga
        cursor.execute("TRUNCATE TABLE staging.orders;")
        
        # Move os dados para o buffer
        csv_buffer = io.StringIO()
        df.to_csv(csv_buffer, index=False, header=True)
        csv_buffer.seek(0)
        
        # Executa o COPY para a tabela
        copy_sql = """
            COPY staging.orders (
                OrderID,
                CustomerID,
                EmployeeID,
                OrderDate,
                RequiredDate,
                ShippedDate,
                ShipVia,
                Freight,
                ShipName,
                ShipAddress,
                ShipCity,
                ShipRegion,
                ShipPostalCode,
                ShipCountry,
                origem
            ) 
            FROM STDIN WITH CSV HEADER
        """
        cursor.copy_expert(copy_sql, csv_buffer)
        conn.commit()